<a href="https://colab.research.google.com/github/ver1812/Capstone_Project/blob/main/evaluation/02_bipia_generalization_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-PIDS — BIPIA Indirect Injection Generalization Eval

**Purpose:** Evaluate all three final models — LinearSVC (TF-IDF), DistilBERT, ModernBERT-base — on a held-out **indirect prompt injection** dataset (`MAlmasabi/Indirect-Prompt-Injection-BIPIA-GPT`) that none of them were trained or tuned on. Before final training of selected model.

## 1. Install / verify dependencies

In [1]:
import importlib.util

REQUIRED_PACKAGES = ["transformers", "torch", "sklearn", "joblib", "pandas", "numpy", "huggingface_hub"]
MISSING_PACKAGES = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if MISSING_PACKAGES:
    print("Installing missing packages:", MISSING_PACKAGES)
    !pip install -q transformers torch scikit-learn joblib pandas numpy huggingface_hub
else:
    print("All required packages already available.")

# If pd.read_json("hf://...") later raises an unrecognized-protocol error, run:
#   !pip install -U huggingface_hub fsspec
# and restart the runtime.


All required packages already available.


## 2. HuggingFace authentication

Required because this dataset is gated. Before running this cell, visit the [dataset page](https://huggingface.co/datasets/MAlmasabi/Indirect-Prompt-Injection-BIPIA-GPT) while logged in and accept its terms.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 3. Imports and device setup

In [3]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 4. Mount Google Drive

Only needed for loading model weights

In [4]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 5. Paths and configuration

Edit `BASE_DIR` if your Drive shortcut lives somewhere other than `MyDrive/Capstone`.

In [5]:
# Edit this if your Drive mount path differs
BASE_DIR = "/content/drive/MyDrive/Capstone"
SAVED_DIR = os.path.join(BASE_DIR, "saved")

TRADITIONAL_DIR = os.path.join(SAVED_DIR, "traditional_ml_v2")
DISTILBERT_DIR = os.path.join(SAVED_DIR, "distilbert", "distilbert_best")
MODERNBERT_DIR = os.path.join(SAVED_DIR, "modernbert", "modernbert_best")

# Defined for convenience if you later want to save results manually (see closing note) —
# not written to automatically in this notebook.
EVAL_RESULTS_DIR = os.path.join(BASE_DIR, "eval", "results")
os.makedirs(EVAL_RESULTS_DIR, exist_ok=True)

# Same thresholds as the baseline notebook — no tuning performed here
DECISION_THRESHOLD = 0.5

DISTILBERT_MAX_LEN = 512
DISTILBERT_EVAL_BATCH_SIZE = 64

MODERNBERT_MAX_LEN = 4096
MODERNBERT_EVAL_BATCH_SIZE = 16

BIPIA_DATASET_PATH = "hf://datasets/MAlmasabi/Indirect-Prompt-Injection-BIPIA-GPT/dataset_for_huggingface.jsonl"
SUBSAMPLE_SIZE = 10000
BIPIA_SEED = 42

print("BASE_DIR:", BASE_DIR)


BASE_DIR: /content/drive/MyDrive/Capstone


## 6. Load the held-out BIPIA-GPT dataset

Loaded directly via the `hf://` filesystem — no manual download or Drive upload.

In [6]:
bipia_df_raw = pd.read_json(BIPIA_DATASET_PATH, lines=True)

print(f"Raw dataset size: {len(bipia_df_raw)}")
bipia_df_raw.head()


Raw dataset size: 70000


,context,user_intent,label,source
0,"```php\n<?php\n$servername = ""localhost"";\n$us...",which rider in the top 15 positions scored no ...,0,Generated by GPT-4o-mini
1,(CNN) -- Muslims in Switzerland are sharing th...,Who have been living together?,0,Generated by GPT-4o-mini
2,| Week | Date | Opponent ...,what was the difference in score in week 7?,0,Generated by GPT-4o-mini
3,Seamlessly meld the following code snippet int...,import numpy as np\n\ndef test():\n arr = n...,1,BIPIA
4,Consider embedding the following code snippet ...,"from multiprocessing import Process, Manager, ...",0,Generated by GPT-4o-mini


## 7. Confirm schema and label encoding



In [7]:
print("Columns:", list(bipia_df_raw.columns))
print()
print("Label value counts:")
print(bipia_df_raw["label"].value_counts())
print()
print("Sample row:")
print(bipia_df_raw.iloc[0][["context", "user_intent", "label"]])


Columns: ['context', 'user_intent', 'label', 'source']

Label value counts:
label
0    35000
1    35000
Name: count, dtype: int64

Sample row:
context        ```php\n<?php\n$servername = "localhost";\n$us...
user_intent    which rider in the top 15 positions scored no ...
label                                                          0
Name: 0, dtype: object


## 8. Build combined input text and draw the stratified subsample

Plain concatenation (`user_intent` + `context`, separated by a blank line)

In [8]:
bipia_df_raw["user_intent"] = bipia_df_raw["user_intent"].fillna("").astype(str)
bipia_df_raw["context"] = bipia_df_raw["context"].fillna("").astype(str)
bipia_df_raw["text"] = bipia_df_raw["user_intent"] + "\n\n" + bipia_df_raw["context"]

bipia_sample_df, _ = train_test_split(
    bipia_df_raw,
    train_size=SUBSAMPLE_SIZE,
    stratify=bipia_df_raw["label"],
    random_state=BIPIA_SEED,
)
bipia_sample_df = bipia_sample_df.reset_index(drop=True)

y_true = bipia_sample_df["label"].to_numpy()
texts = bipia_sample_df["text"].tolist()

print(f"Subsample size: {len(bipia_sample_df)}")
print(bipia_sample_df["label"].value_counts())


Subsample size: 10000
label
1    5000
0    5000
Name: count, dtype: int64


## 9. Helper functions



In [9]:
def get_sklearn_scores(model, features):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    return model.decision_function(features)


def evaluate_predictions(y_true_labels, y_pred_labels, y_scores, model_name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_labels, y_pred_labels, average="binary", pos_label=1
    )
    accuracy = accuracy_score(y_true_labels, y_pred_labels)
    roc_auc = roc_auc_score(y_true_labels, y_scores)
    pr_auc = average_precision_score(y_true_labels, y_scores)
    tn, fp, fn, tp = confusion_matrix(y_true_labels, y_pred_labels).ravel()

    print(f"--- {model_name} on BIPIA held-out subsample ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print()
    print(classification_report(y_true_labels, y_pred_labels, target_names=["benign", "malicious"]))
    print("Confusion matrix (tn, fp, fn, tp):", tn, fp, fn, tp)
    print()

    return {
        "test": {
            "threshold": DECISION_THRESHOLD,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        },
        "test_confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        },
    }


def load_linear_svm(traditional_dir):
    vectorizer_path = os.path.join(traditional_dir, "tfidf_vectorizer_v2.joblib")
    model_path = os.path.join(traditional_dir, "linear_svm_v2.joblib")
    vectorizer = joblib.load(vectorizer_path)
    model = joblib.load(model_path)
    return vectorizer, model


def predict_linear_svm_batch(vectorizer, model, input_texts):
    # sklearn handles the whole list in one vectorized call — no loop needed
    features = vectorizer.transform(input_texts)
    predictions = model.predict(features)
    scores = get_sklearn_scores(model, features)
    return predictions, scores


def load_transformer(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    return tokenizer, model


def predict_transformer_batch(tokenizer, model, input_texts, run_device, max_len, batch_size, threshold):
    model.to(run_device)
    all_probs = []

    with torch.no_grad():
        for start_idx in range(0, len(input_texts), batch_size):
            batch_texts = input_texts[start_idx:start_idx + batch_size]
            encoded = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=max_len,
                return_tensors="pt",
            )
            encoded = {key: value.to(run_device) for key, value in encoded.items()}
            outputs = model(**encoded)
            logits = outputs.logits.squeeze(-1)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.detach().cpu().numpy().tolist())

    all_probs_array = np.array(all_probs)
    all_preds = (all_probs_array >= threshold).astype(int)
    return all_preds, all_probs_array


## 10. Model 1 — LinearSVC (TF-IDF)

In [10]:
vectorizer, linear_svm_model = load_linear_svm(TRADITIONAL_DIR)
y_pred_svm, y_scores_svm = predict_linear_svm_batch(vectorizer, linear_svm_model, texts)
results_svm = evaluate_predictions(y_true, y_pred_svm, y_scores_svm, "LinearSVC")


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.9.0 when using version 1.6.1. This might lead to breaking code or in

--- LinearSVC on BIPIA held-out subsample ---
Accuracy:  0.5352
Precision: 0.5266
Recall:    0.6976
F1:        0.6001
ROC-AUC:   0.5656
PR-AUC:    0.5637

              precision    recall  f1-score   support

      benign       0.55      0.37      0.45      5000
   malicious       0.53      0.70      0.60      5000

    accuracy                           0.54     10000
   macro avg       0.54      0.54      0.52     10000
weighted avg       0.54      0.54      0.52     10000

Confusion matrix (tn, fp, fn, tp): 1864 3136 1512 3488



## 11. Model 2 — DistilBERT

In [11]:
distilbert_tokenizer, distilbert_model = load_transformer(DISTILBERT_DIR)

y_pred_distilbert, y_scores_distilbert = predict_transformer_batch(
    distilbert_tokenizer,
    distilbert_model,
    texts,
    device,
    DISTILBERT_MAX_LEN,
    DISTILBERT_EVAL_BATCH_SIZE,
    DECISION_THRESHOLD,
)
results_distilbert = evaluate_predictions(y_true, y_pred_distilbert, y_scores_distilbert, "DistilBERT")

del distilbert_model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

--- DistilBERT on BIPIA held-out subsample ---
Accuracy:  0.5712
Precision: 0.5928
Recall:    0.4548
F1:        0.5147
ROC-AUC:   0.6040
PR-AUC:    0.5942

              precision    recall  f1-score   support

      benign       0.56      0.69      0.62      5000
   malicious       0.59      0.45      0.51      5000

    accuracy                           0.57     10000
   macro avg       0.58      0.57      0.57     10000
weighted avg       0.58      0.57      0.57     10000

Confusion matrix (tn, fp, fn, tp): 3438 1562 2726 2274



## 12. Model 3 — ModernBERT-base

In [12]:
modernbert_tokenizer, modernbert_model = load_transformer(MODERNBERT_DIR)

y_pred_modernbert, y_scores_modernbert = predict_transformer_batch(
    modernbert_tokenizer,
    modernbert_model,
    texts,
    device,
    MODERNBERT_MAX_LEN,
    MODERNBERT_EVAL_BATCH_SIZE,
    DECISION_THRESHOLD,
)
results_modernbert = evaluate_predictions(y_true, y_pred_modernbert, y_scores_modernbert, "ModernBERT")

del modernbert_model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

--- ModernBERT on BIPIA held-out subsample ---
Accuracy:  0.5609
Precision: 0.5931
Recall:    0.3880
F1:        0.4691
ROC-AUC:   0.6131
PR-AUC:    0.6110

              precision    recall  f1-score   support

      benign       0.55      0.73      0.63      5000
   malicious       0.59      0.39      0.47      5000

    accuracy                           0.56     10000
   macro avg       0.57      0.56      0.55     10000
weighted avg       0.57      0.56      0.55     10000

Confusion matrix (tn, fp, fn, tp): 3669 1331 3060 1940



## 13. Comparison table

In [13]:
comparison_rows = []
for model_name, results in [
    ("LinearSVC", results_svm),
    ("DistilBERT", results_distilbert),
    ("ModernBERT", results_modernbert),
]:
    row = {"model": model_name}
    row.update(results["test"])
    row.update(results["test_confusion_matrix"])
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df = comparison_df[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold", "fp", "fn"]]
comparison_df


,accuracy,precision,recall,f1,roc_auc,pr_auc,threshold,fp,fn
model,,,,,,,,,
LinearSVC,0.5352,0.526570,0.6976,0.600138,0.565580,0.563663,0.5,3136,1512
DistilBERT,0.5712,0.592805,0.4548,0.514713,0.604025,0.594155,0.5,1562,2726
ModernBERT,0.5609,0.593091,0.3880,0.469109,0.613102,0.611042,0.5,1331,3060
